In [ ]:
pip install zstandard

In [ ]:
import pandas as pd

In [ ]:
import zstandard as zstd
import json
import io

file_path = "RS_2019-04.zst"

posts = []

with open(file_path, 'rb') as fh:
    dctx = zstd.ZstdDecompressor()
    stream_reader = dctx.stream_reader(fh)
    text_stream = io.TextIOWrapper(stream_reader, encoding='utf-8')

    for line in text_stream:
        try:
            post = json.loads(line)
            body = post.get("selftext", "").strip()
            
            # Skip if body is blank, deleted, or removed
            if not body or body == "[deleted]" or body == "[removed]":
                continue
            
            posts.append({
                "id": post.get("id"),
                "author": post.get("author"),
                "subreddit": post.get("subreddit"),
                "title": post.get("title", ""),
                "body": body,
                "score": post.get("score"),
                "num_comments": post.get("num_comments", 0),
                "created_utc": post.get("created_utc")
            })

            if len(posts) > 500000:
                break

        except:
            pass

df = pd.DataFrame(posts)

print(df.head())


## Preprocess the data 


In [ ]:
# any any of the jibris symbols and making the data clean and ready for the model
df['body'] = df['body'].str.replace(r'[^\x00-\x7F]+', ' ', regex=True)

In [ ]:
blank_body_count = df["body"].fillna("").str.strip().eq("").sum()
non_blank_body_count = df["body"].fillna("").str.strip().ne("").sum()

print("Blank body rows:", blank_body_count)
print("Non-blank body rows:", non_blank_body_count)

In [ ]:
import re
import html

def normalize_text(text):
    text = html.unescape(str(text))          # decode &amp;, &lt;, etc.
    text = text.replace("\\n", " ")          # literal \n
    text = text.replace("\n", " ")           # real newline
    text = text.replace("\r", " ").replace("\t", " ")
    text = text.replace("*", "").replace('""', '') # *
    text = re.sub(r"\s+", " ", text).strip() # collapse extra spaces
    return text

df["body"] = df["body"].apply(normalize_text)

df.head()

In [ ]:

df["created_utc"] = pd.to_datetime(df["created_utc"], unit="s")

In [ ]:
# ok so the subreddits is just like tthe overall catogory 
df["subreddit"].value_counts().head(20)


In [ ]:
df = df[~df["subreddit"].str.startswith("u_")]

In [ ]:
df["subreddit"].value_counts().head(40)


In [ ]:
df = df[df["subreddit"] != "removalbot"]

In [ ]:
df["subreddit"].value_counts()
df['subreddit'].to_csv("subreddits.csv", index=False)


In [ ]:
df

In [ ]:
import numpy as np
print(" FEATURE ENGINEERING LAYER")

# Engagement features
df["engagement_score"] = np.log1p(df["score"] + 1)  # Log-transform to reduce skew])
df["post_length"] = df["body"].str.len()
df["word_count"] = df["body"].str.split().str.len()

# Time features
ref_time = df["created_utc"].max()
df["hours_ago"] = (ref_time - df["created_utc"]).dt.total_seconds() / 3600
df["recency_weight"] = np.exp(-0.0001 * df["hours_ago"])  # Exponential decay

# Extract hour and day of week
df["hour_posted"] = df["created_utc"].dt.hour
df["day_of_week"] = df["created_utc"].dt.dayofweek

# Create interaction pairs (for collaborative filtering)
df["user_post_id"] = df["author"] + "_" + df["id"]


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set seaborn style
sns.set_style("whitegrid")
sns.set_palette("husl")

## 📊 Data Visualization & Analysis

This section contains comprehensive visualizations using **Seaborn** to understand the data:

### Before Training:
1. **Feature Distributions** - Engagement scores, post lengths, word counts, recency weights
2. **Temporal Patterns** - Posts by hour of day and day of week
3. **Feature Correlations** - Heatmap showing relationships between numerical features
4. **Top Subreddits** - Most active subreddits by post count

### During Training:
5. **Enforced Categories** - Distribution of keyword-based categories
6. **KMeans Elbow Method** - Finding optimal number of clusters
7. **Final Category Distribution** - Combined enforced + auto-clustered categories
8. **User-Category Interaction Matrix** - Heatmap of user engagement across categories

### After Training:
9. **Engagement by Category** - Box plots showing engagement distribution
10. **Feature Pairplot** - Scatter matrix of key numerical features

In [ ]:
# Visualizations BEFORE Training

# 1. Distribution of engagement scores
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Engagement score distribution
sns.histplot(data=df, x='engagement_score', bins=50, kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution of Engagement Scores', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Engagement Score (log-transformed)')

# Post length distribution
sns.histplot(data=df, x='post_length', bins=50, kde=True, ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Distribution of Post Lengths', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Post Length (characters)')

# Word count distribution
sns.histplot(data=df, x='word_count', bins=50, kde=True, ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Distribution of Word Counts', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Word Count')

# Recency weight distribution  
sns.histplot(data=df, x='recency_weight', bins=50, kde=True, ax=axes[1, 1], color='plum')
axes[1, 1].set_title('Distribution of Recency Weights', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Recency Weight (exponential decay)')

plt.tight_layout()
plt.show()

In [ ]:
# 2. Temporal patterns visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Posts by hour of day
hour_counts = df['hour_posted'].value_counts().sort_index()
sns.barplot(x=hour_counts.index, y=hour_counts.values, ax=axes[0], palette='viridis')
axes[0].set_title('Posts Distribution by Hour of Day', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Hour (24-hour format)')
axes[0].set_ylabel('Number of Posts')

# Posts by day of week
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_counts = df['day_of_week'].value_counts().sort_index()
sns.barplot(x=[day_names[i] for i in day_counts.index], y=day_counts.values, ax=axes[1], palette='coolwarm')
axes[1].set_title('Posts Distribution by Day of Week', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Number of Posts')

plt.tight_layout()
plt.show()

In [ ]:
# 3. Feature correlation heatmap
numerical_features = ['score', 'num_comments', 'engagement_score', 'post_length', 
                      'word_count', 'hours_ago', 'recency_weight', 'hour_posted', 'day_of_week']

plt.figure(figsize=(12, 8))
correlation_matrix = df[numerical_features].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Top subreddits visualization
plt.figure(figsize=(14, 6))
top_subreddits = df['subreddit'].value_counts().head(20)
sns.barplot(x=top_subreddits.values, y=top_subreddits.index, palette='rocket')
plt.title('Top 20 Subreddits by Post Count', fontsize=16, fontweight='bold')
plt.xlabel('Number of Posts')
plt.ylabel('Subreddit')
plt.tight_layout()
plt.show()

In [ ]:
import os
from sentence_transformers import SentenceTransformer

# Build combined text once (needed by the next cell)
df["full_text"] = (df["title"].fillna("") + " " + df["body"].fillna("")).str.strip()

# Check if embedding model already exists
model_path = "embedding_model"
if os.path.exists(model_path):
    print(f"Loading existing model from {model_path}...")
    embedding_model = SentenceTransformer(model_path)
else:
    print("Loading new model: paraphrase-MiniLM-L3-v2")
    embedding_model = SentenceTransformer("paraphrase-MiniLM-L3-v2")

    chunk_size = 20000
    all_embeddings = []

    for i in range(0, len(df), chunk_size):
        batch_texts = df["full_text"].iloc[i:i + chunk_size].tolist()
        batch_embeddings = embedding_model.encode(
            batch_texts,
            batch_size=128,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")
        all_embeddings.append(batch_embeddings)

    embeddings = np.vstack(all_embeddings)
    df["text_embedding"] = list(embeddings)

In [ ]:
# save the embedding_model
embedding_model.save("embedding_model")

In [ ]:
# enforcing categories and using kmeans to cluster the remaining posts

from sklearn.cluster import KMeans

# Build combined text (if not already created in previous cell)
if "full_text" not in df.columns:
    df["full_text"] = (df["title"].fillna("") + " " + df["body"].fillna("")).str.strip()

# 15 enforced keyword-based categories
ENFORCED_CATEGORIES  = {
    "gaming": [
        "fortnite", "apex", "league", "rocketleague", "dnd", "minecraft", "valorant", 
        "csgo", "cod", "call of duty", "pubg", "battlefield", "overwatch", "elden ring",
        "zelda", "skyrim", "assassins creed", "rpg", "moba", "gamer", "gaming", "steam",
        "xbox", "playstation", "nintendo", "switch", "fps", "esports", "multiplayer"
    ],
    "relationships": [
        "relationship", "dating", "love", "breakup", "marriage", "divorce", "boyfriend", 
        "girlfriend", "partner", "crush", "flirt", "romance", "friendship", "toxic", 
        "conflict", "communication", "jealousy", "advice", "trust", "feelings", "proposal"
    ],
    "career_jobs": [
        "job", "career", "resume", "interview", "application", "hiring", "promotion", 
        "salary", "internship", "work", "freelance", "contract", "office", "manager", 
        "boss", "colleague", "employment", "profession", "networking", "skills", "job search"
    ],
    "education": [
        "school", "college", "university", "exam", "study", "assignment", "homework", 
        "research", "grades", "test", "lecture", "professor", "degree", "class", 
        "course", "tuition", "scholarship", "study tips", "student", "learning"
    ],
    "finance": [
        "money", "stocks", "crypto", "budget", "finance", "saving", "investment", 
        "bitcoin", "ethereum", "loan", "debt", "bank", "credit", "tax", "retirement", 
        "portfolio", "dividends", "trading", "economy", "financial planning"
    ],
    "technology": [
        "tech", "programming", "buildapc", "software", "hardware", "ai", "machine learning",
        "coding", "python", "javascript", "java", "react", "node", "data science", 
        "cloud", "servers", "network", "security", "blockchain", "gadgets", "computers"
    ],
    "entertainment": [
        "movie", "tv", "series", "netflix", "disney", "hollywood", "bollywood", "music", 
        "concert", "show", "theater", "celebrity", "actors", "actress", "gaming", 
        "streaming", "youtube", "tiktok", "reviews", "reviews", "entertainment"
    ],
    "mental_health": [
        "depression", "anxiety", "stress", "mental health", "therapy", "counseling", 
        "self care", "mindfulness", "panic", "suicide", "help", "support", "PTSD", 
        "emotional", "meditation", "motivation", "overthinking", "burnout", "mental illness", "coping"
    ],
    "parenting_family": [
        "parent", "family", "kids", "children", "baby", "toddler", "adolescent", "school", 
        "homework", "education", "mom", "dad", "siblings", "grandparent", "household", 
        "childcare", "discipline", "pregnancy", "family time", "relationship"
    ],
    "health_fitness": [
        "exercise", "workout", "gym", "running", "diet", "nutrition", "weight loss", 
        "fitness", "cardio", "strength", "training", "yoga", "pilates", "health", "vitamins", 
        "supplements", "wellness", "endurance", "sports", "healthy"
    ],
    "travel": [
        "trip", "vacation", "tourism", "flight", "hotel", "backpacking", "road trip", 
        "adventure", "itinerary", "guide", "beach", "mountains", "travel tips", "visa", 
        "passport", "travel blog", "culture", "explore", "destination", "journey"
    ],
    "sports": [
        "football", "soccer", "basketball", "tennis", "cricket", "baseball", "hockey", 
        "olympics", "athletics", "golf", "rugby", "coach", "team", "match", "league", 
        "training", "tournament", "player", "score", "competition"
    ],
    "news_politics": [
        "news", "politics", "election", "government", "policy", "vote", "campaign", 
        "president", "prime minister", "congress", "senate", "political party", 
        "debate", "protest", "law", "bill", "diplomacy", "international", "political news", "current affairs"
    ],
    "food_cooking": [
        "recipe", "cook", "cooking", "baking", "meal", "restaurant", "ingredients", 
        "food", "dinner", "lunch", "breakfast", "healthy eating", "chef", "dessert", 
        "grill", "vegetarian", "vegan", "snack", "kitchen", "cuisine"
    ],
    "science": [
        "physics", "chemistry", "biology", "research", "experiment", "laboratory", 
        "space", "astronomy", "earth", "math", "quantum", "genetics", "technology", 
        "environment", "climate", "scientist", "study", "discovery", "innovation", "theory"
    ]
}

In [ ]:


def assign_enforced_category(text: str):
    t = str(text).lower()
    for category, keywords in ENFORCED_CATEGORIES.items():
        if any(kw in t for kw in keywords):
            return category
    return None

# Enforced assignment

df["enforced_category"] = df["full_text"].apply(assign_enforced_category)


In [ ]:
# Before clustering: Show enforced category distribution
enforced_counts = df['enforced_category'].value_counts()
print(f"Posts with enforced categories: {enforced_counts.sum()}")
print(f"Posts without enforced categories: {df['enforced_category'].isna().sum()}")

plt.figure(figsize=(12, 6))
sns.barplot(x=enforced_counts.values, y=enforced_counts.index, palette='Set2')
plt.title('Distribution of Enforced Categories (Before Auto-Clustering)', fontsize=16, fontweight='bold')
plt.xlabel('Number of Posts')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

In [ ]:

target_total_categories = 10
used_enforced = sorted(df["enforced_category"].dropna().unique().tolist())
used_enforced_count = len(used_enforced)

remaining_mask = df["enforced_category"].isna()
remaining_count = int(remaining_mask.sum())

n_auto = max(0, target_total_categories - used_enforced_count)

if n_auto > 0 and remaining_count > 0:
    n_clusters = min(n_auto, remaining_count)
    remaining_embeddings = embeddings[remaining_mask.to_numpy()]

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_ids = kmeans.fit_predict(remaining_embeddings)

    df.loc[remaining_mask, "auto_category"] = [f"auto_cluster_{i}" for i in cluster_ids]
else:
    df["auto_category"] = None



In [ ]:
# During Training: Elbow method for KMeans
from sklearn.cluster import KMeans

print("DURING TRAINING: Finding optimal number of clusters...")

target_total_categories = 10
used_enforced = sorted(df["enforced_category"].dropna().unique().tolist())
used_enforced_count = len(used_enforced)

remaining_mask = df["enforced_category"].isna()
remaining_count = int(remaining_mask.sum())

n_auto = max(0, target_total_categories - used_enforced_count)

if n_auto > 0 and remaining_count > 0:
    n_clusters = min(n_auto, remaining_count)
    remaining_embeddings = embeddings[remaining_mask.to_numpy()]
    
    # Elbow method to visualize cluster quality
    max_k = min(10, remaining_count)
    inertias = []
    K_range = range(1, max_k + 1)
    
    print(f"Testing {max_k} different cluster sizes...")
    for k in K_range:
        kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans_temp.fit(remaining_embeddings)
        inertias.append(kmeans_temp.inertia_)
    
    # Plot elbow curve
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=list(K_range), y=inertias, marker='o', linewidth=2, markersize=8, color='dodgerblue')
    plt.axvline(x=n_clusters, color='red', linestyle='--', linewidth=2, label=f'Selected: {n_clusters} clusters')
    plt.title('KMeans Elbow Method: Finding Optimal Clusters', fontsize=16, fontweight='bold')
    plt.xlabel('Number of Clusters (k)', fontsize=12)
    plt.ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
    plt.xticks(K_range)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\n Selected {n_clusters} auto-clusters (to complement {used_enforced_count} enforced categories)")
else:
    print("No auto-clustering needed")

In [ ]:

df["final_category"] = df["enforced_category"].fillna(df["auto_category"]).fillna("uncategorized")

# Create user × category interaction matrix
interaction_matrix = df.pivot_table(
    index="author",
    columns="final_category",
    values="engagement_score",
    aggfunc="sum",
    fill_value=0.0,
)


In [ ]:
# After Clustering: Visualize Final Category Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Final category distribution
final_counts = df['final_category'].value_counts().head(15)
sns.barplot(x=final_counts.values, y=final_counts.index, ax=axes[0], palette='mako')
axes[0].set_title('Top 15 Final Categories Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Posts')
axes[0].set_ylabel('Category')

# Pie chart of category types
category_types = pd.Series({
    'Enforced Categories': df['enforced_category'].notna().sum(),
    'Auto-Clustered': df['auto_category'].notna().sum(),
    'Uncategorized': (df['final_category'] == 'uncategorized').sum()
})
colors = ['#FF6B6B', '#4ECDC4', '#FFE66D']
axes[1].pie(category_types.values, labels=category_types.index, autopct='%1.1f%%',
            startangle=90, colors=colors, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Category Assignment Breakdown', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 Categorization Summary:")
print(f"  • Enforced categories: {df['enforced_category'].notna().sum()} posts")
print(f"  • Auto-clustered: {df['auto_category'].notna().sum()} posts")
print(f"  • Uncategorized: {(df['final_category'] == 'uncategorized').sum()} posts")
print(f"  • Total categories: {df['final_category'].nunique()}")

In [ ]:
# User-Category Interaction Matrix Preview
plt.figure(figsize=(14, 8))

# Show a sample of the interaction matrix (first 30 users, all categories)
sample_size = min(30, len(interaction_matrix))
sample_matrix = interaction_matrix.head(sample_size)

sns.heatmap(sample_matrix, cmap='YlOrRd', cbar_kws={'label': 'Engagement Score'},
            linewidths=0.5, linecolor='white')
plt.title(f'User-Category Interaction Heatmap (First {sample_size} Users)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Category', fontsize=12)
plt.ylabel('User (Author)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Interaction Matrix Shape: {interaction_matrix.shape}")
print(f"Total unique users: {len(interaction_matrix)}")
print(f"Total categories: {len(interaction_matrix.columns)}")

In [ ]:
# Engagement Score by Category (Box Plot)
plt.figure(figsize=(14, 8))

# Get top categories for clearer visualization
top_categories = df['final_category'].value_counts().head(12).index
df_top = df[df['final_category'].isin(top_categories)]

sns.boxplot(data=df_top, y='final_category', x='engagement_score', palette='Set3')
plt.title('Engagement Score Distribution by Category', fontsize=16, fontweight='bold')
plt.xlabel('Engagement Score (log-transformed)', fontsize=12)
plt.ylabel('Category', fontsize=12)
plt.tight_layout()
plt.show()

# Show mean engagement by category
mean_engagement = df.groupby('final_category')['engagement_score'].mean().sort_values(ascending=False).head(10)
print("\n🏆 Top 10 Categories by Average Engagement:")
for cat, score in mean_engagement.items():
    print(f"  {cat}: {score:.3f}")

In [ ]:
# Advanced: Pairplot of key numerical features
print("Creating pairplot of key features (this may take a moment)...")

# Sample data for faster plotting
sample_df = df.sample(n=min(5000, len(df)), random_state=42)
features_for_pairplot = ['engagement_score', 'post_length', 'word_count', 'recency_weight']

sns.pairplot(sample_df[features_for_pairplot], 
             diag_kind='kde', 
             plot_kws={'alpha': 0.6, 's': 30},
             diag_kws={'alpha': 0.7})
plt.suptitle('Pairplot of Key Features', y=1.02, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(" All visualizations complete!")

In [ ]:

# Save both outputs to pickle
processed_path = "processed_posts.pkl"
matrix_path = "user_category_interaction_matrix.pkl"

df.to_pickle(processed_path)
interaction_matrix.to_pickle(matrix_path)



In [ ]:
print("Processed DataFrame shape:", df.shape)
print("Interaction matrix shape:", interaction_matrix.shape)
print("Final category count:", df["final_category"].nunique())
print("Saved:", processed_path)
print("Saved:", matrix_path)
print(df["final_category"].value_counts().head(20))

In [ ]:
df.info()

In [ ]:
cols_to_drop = ["author", "hours_ago", "user_post_id", "auto_category", "enforced_category"]
df = df.drop(columns=cols_to_drop)

In [ ]:
embeddings = np.array(embeddings).astype("float32")

In [ ]:
import faiss
# 4. Create FAISS index
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)


In [ ]:
# 5. Add embeddings to FAISS
index.add(embeddings)

# 6. Save FAISS index
faiss.write_index(index, "reddit_posts.faiss")

In [ ]:
embedding_model = SentenceTransformer("paraphrase-MiniLM-L3-v2")
query = "transformer model released"
query_vector = embedding_model.encode([query]).astype("float32")

D, I = index.search(query_vector, k=10)
print(I)  # indices of top 10 similar posts

In [ ]:
# POST_DB Schema:
post_db_schema = {
    "post_id": "string (unique subreddit post ID)",
    "subreddit": "string (original subreddit, e.g., 'technology')",
    "title": "string (post title)",
    "body": "string (cleaned post content)",
    "score": "int (upvotes - downvotes)",
    "num_comments": "int (number of comments)",
    "created_utc": "datetime (when post was created)",
    "engagement_score": "float (log-transformed score for weighting)",
    "post_length": "int (character count of body)",
    "word_count": "int (word count)",
    "hour_posted": "int (0-23, hour of day)",
    "day_of_week": "int (0-6, Monday=0)",
    "recency_weight": "float (exponential decay based on age)",
    "final_category": "string (gaming|relationships|career_jobs|...|uncategorized)",
    "full_text": "string (title + body combined)",
    "text_embedding": "ndarray[384] (dense vector from SentenceTransformer)"
}

print("POST_DB has", len(df), "posts")
print("POST_DB columns:", df.columns.tolist())
print("\nSample post:\n", df.iloc[0])


In [ ]:
# SAVE MODEL ARTIFACTS (what to persist)
import os
import pickle

def save_model_artifacts():
    """Save all model components for inference without retraining."""
    
    artifacts = {
        # 1. Embedding Model (transformer weights)
        "embedding_model_path": "embedding_model/",
        
        # 2. FAISS Index (semantic search structure)
        "faiss_index_path": "reddit_posts.faiss",
        
        # 3. Processed Posts DataFrame (metadata + embeddings)
        "posts_df_path": "processed_posts.pkl",
        
        # 4. User-Category Interaction Matrix (for collab filtering)
        "interaction_matrix_path": "user_category_interaction_matrix.pkl",
    }
    
    print("MODEL ARTIFACTS TO SAVE:")
    for name, path in artifacts.items():
        if os.path.exists(path):
            size = os.path.getsize(path) if os.path.isfile(path) else "directory"
            print(f"  {name}: {path} ({size})")
        else:
            print(f"  {name}: {path} (not yet saved)")
    
    return artifacts

artifacts = save_model_artifacts()

print("\nThese files are already saved (check earlier cells)")
print(" Keep them in the MODEL/ directory for the backend to load")


In [ ]:
# LOAD ARTIFACTS FOR BACKEND INFERENCE

def load_model_artifacts():
    """Load all pre-trained artifacts for inference."""
    from sentence_transformers import SentenceTransformer
    import pickle
    
    print("Loading model artifacts...")
    
    # 1. Load embedding model
    embedding_model = SentenceTransformer("embedding_model")
    print(" Loaded embedding model")
    
    # 2. Load FAISS index
    faiss_index = faiss.read_index("reddit_posts.faiss")
    print(f"Loaded FAISS index ({faiss_index.ntotal} posts)")
    
    # 3. Load posts DataFrame (contains metadata)
    with open("processed_posts.pkl", "rb") as f:
        posts_df = pickle.load(f)
    print(f"Loaded posts DataFrame ({len(posts_df)} posts)")
    
    # 4. Load interaction matrix
    with open("user_category_interaction_matrix.pkl", "rb") as f:
        interaction_matrix = pickle.load(f)
    print(f"Loaded interaction matrix ({interaction_matrix.shape})")
    
    return {
        "embedding_model": embedding_model,
        "faiss_index": faiss_index,
        "posts_df": posts_df,
        "interaction_matrix": interaction_matrix
    }

# Load artifacts (only if files exist)
if os.path.exists("embedding_model") and os.path.exists("reddit_posts.faiss"):
    artifacts_loaded = load_model_artifacts()
    embedding_model = artifacts_loaded["embedding_model"]
    faiss_index = artifacts_loaded["faiss_index"]
    posts_df = artifacts_loaded["posts_df"]
    interaction_matrix = artifacts_loaded["interaction_matrix"]
else:
    print("Model artifacts not found. Run earlier cells to generate them.")
    artifacts_loaded = None


## 🔧 Embedding Model Finetuning

Here we finetune `paraphrase-MiniLM-L3-v2` on our Reddit post data using **contrastive learning**.

**Strategy**: We use `MultipleNegativesRankingLoss` (MNRL) with `(anchor, positive)` pairs built from posts in the same `final_category`.
- **Anchor**: Post A's `full_text`
- **Positive**: Post B's `full_text` from the same category
- **Negatives**: All other posts in the batch (in-batch negatives — free!)

This teaches the model that posts about the same topic should have similar embeddings — improving downstream FAISS retrieval quality.


In [ ]:
import random
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

#  Config 
BASE_MODEL      = "paraphrase-MiniLM-L3-v2"   # same model used in main pipeline
OUTPUT_DIR      = "embedding_model_finetuned"            # overwrites the existing saved model
PAIRS_PER_CAT   = 3_000    # anchor-positive pairs sampled per category
MAX_SEQ_LEN     = 128      # token limit (MiniLM-L3 is fast; keep short for speed)
BATCH_SIZE      = 64
EPOCHS          = 10        # 1 epoch is usually enough with MNRL on large data
WARMUP_RATIO    = 0.1      # fraction of steps used for LR warm-up
RANDOM_SEED     = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print("Config OK")


In [ ]:
#  Build anchor-positive pairs 
# df must already have 'full_text' and 'final_category' columns (from earlier cells)

assert "full_text" in df.columns, "Run earlier cells first to create 'full_text'"
assert "final_category" in df.columns, "Run earlier cells first to create 'final_category'"

# Filter out uncategorized posts — they add noise rather than signal
df_cat = df[df["final_category"] != "uncategorized"].copy()
df_cat = df_cat[df_cat["full_text"].str.len() > 30]  # drop very short posts

train_examples = []

for category, group in df_cat.groupby("final_category"):
    texts = group["full_text"].tolist()
    if len(texts) < 2:
        continue

    n_pairs = min(PAIRS_PER_CAT, len(texts) // 2)
    anchors  = random.sample(texts, n_pairs)
    positives = random.sample(texts, n_pairs)

    for a, p in zip(anchors, positives):
        if a != p:                          # skip identical pairs
            train_examples.append(InputExample(texts=[a, p]))

random.shuffle(train_examples)
print(f"Total training pairs: {len(train_examples):,}")
print(f"Example: '{train_examples[0].texts[0][:80]}...'")


In [ ]:
#  Load base model and set up training 

model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = MAX_SEQ_LEN

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
)

# MultipleNegativesRankingLoss:
#   - treats every OTHER pair in the batch as a negative
#   - no need to explicitly label negatives
#   - works great with large diverse batches
train_loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = int(len(train_dataloader) * EPOCHS * WARMUP_RATIO)
print(f"Steps per epoch : {len(train_dataloader)}")
print(f"Warmup steps    : {warmup_steps}")


In [ ]:
# Train 

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path=OUTPUT_DIR,
    show_progress_bar=True,
)

print(f"\n Finetuned model saved to '{OUTPUT_DIR}/'")


In [ ]:
#  Quick sanity-check: compare embedding similarity before vs after 
# Reload the saved finetuned model
OUTPUT_DIR = "embedding_model_finetuned"  # same as used in training cell
finetuned_model = SentenceTransformer(OUTPUT_DIR)
base_model      = SentenceTransformer(BASE_MODEL)

test_pairs = [
    # (same-topic pairs — should be HIGH similarity)
    ("How do I get better at coding interviews?",
     "Tips for cracking technical interviews and landing a software job"),
    ("I'm feeling really depressed and don't know what to do",
     "Struggling with anxiety and mental health, need advice"),
    # (different-topic pair — should be LOW similarity)
    ("Best gaming mouse for FPS games?",
     "My girlfriend broke up with me and I'm devastated"),
]

from sentence_transformers import util

print(f"{'Pair':<10} {'Base sim':>10} {'Finetuned sim':>15}")
print("-" * 40)
for i, (a, b) in enumerate(test_pairs):
    base_sim = util.cos_sim(
        base_model.encode(a), base_model.encode(b)
    ).item()
    ft_sim = util.cos_sim(
        finetuned_model.encode(a), finetuned_model.encode(b)
    ).item()
    label = "(same topic)" if i < 2 else "(diff topic)"
    print(f"Pair {i+1} {label:<14} {base_sim:>8.4f}   {ft_sim:>10.4f}")


In [ ]:
#  Re-embed all posts with the finetuned model and rebuild FAISS index 
# Run this after finetuning to propagate the improved embeddings through the pipeline.

import faiss
from tqdm.auto import tqdm

BATCH = 512
texts = df["full_text"].tolist()

print(f"Re-embedding {len(texts):,} posts with finetuned model...")
embeddings = finetuned_model.encode(
    texts,
    batch_size=BATCH,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype("float32")

# Store back into df
df["text_embedding"] = list(embeddings)

# Rebuild FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
faiss.write_index(index, "reddit_posts.faiss")

# Re-save the processed DataFrame
df.to_pickle("processed_posts.pkl")

print(f"\n FAISS index rebuilt with {index.ntotal:,} vectors (dim={dimension})")
print(" processed_posts.pkl updated")
print(" All model artifacts are now consistent with the finetuned embedder")
